In [23]:
!pip install langchain chromadb openai tiktoken pypdf langchain_openai langchain-community wikipedia faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 37.0 MB/s eta 0:00:00


In [6]:
import os
os.environ["OPENAI_API_KEY"] = "nitishHeBzuaYoA"

Wikipedia Retriever

In [7]:
from langchain_community.retrievers import WikipediaRetriever

In [8]:
# Initialize the retriever (its optional: set language and top_k)
retriever = WikipediaRetriever(top_k_results=3, lang="en")

In [9]:
# Define your query
query = "The impact of quantum mechanics on modern computing technologies"
# Get relevant Wikipedia documents
docs = retriever.invoke(query)

In [10]:
docs

[Document(metadata={'title': 'Timeline of quantum computing and communication', 'summary': 'This is a timeline of quantum computing.\n\n', 'source': 'https://en.wikipedia.org/wiki/Timeline_of_quantum_computing_and_communication'}, page_content='This is a timeline of quantum computing.\n\n\n== 1960s ==\n\n\n=== 1968/69/70 ===\nStephen Wiesner invents conjugate coding\n\n\n=== 1969 ===\n13 June – James L. Park (Washington State University, Pullman)\'s paper is received by Foundations of Physics  in which he describes the non possibility of disturbance in a quantum  transition state in the context of a disproof of quantum jumps in the concept of the atom described by  Bohr. \n\n\n== 1970s ==\n\n\n=== 1973 ===\nAlexander Holevo\'s paper  is published  - the Holevo bound describes a limit of the quantity of classical information which is possible to quanta encode. \nCharles H. Bennett shows that computation can be done reversibly.\n\n\n=== 1975 ===\nR. P. Poplavskii publishes "Thermodynamic

This tells the WikipediaRetriever to:
Go to Wikipedia,
Find pages relevant to your query,
Break the text into chunks (usually paragraphs),
Return them as a list of LangChain Document objects.

Each Document contains:
page_content: The actual text from Wikipedia.
metadata: Info like which Wikipedia page it came from.



In [11]:
# Print retrieved content
for i, doc in enumerate(docs):
    print(f"\n--- Result {i+1} ---")
    print(f"Content:\n{doc.page_content}...")  # truncate for display


--- Result 1 ---
Content:
This is a timeline of quantum computing.


== 1960s ==


=== 1968/69/70 ===
Stephen Wiesner invents conjugate coding


=== 1969 ===
13 June – James L. Park (Washington State University, Pullman)'s paper is received by Foundations of Physics  in which he describes the non possibility of disturbance in a quantum  transition state in the context of a disproof of quantum jumps in the concept of the atom described by  Bohr. 


== 1970s ==


=== 1973 ===
Alexander Holevo's paper  is published  - the Holevo bound describes a limit of the quantity of classical information which is possible to quanta encode. 
Charles H. Bennett shows that computation can be done reversibly.


=== 1975 ===
R. P. Poplavskii publishes "Thermodynamical models of information processing" (in Russian) which shows the computational infeasibility of simulating quantum systems on classical computers, due to the superposition principle.
Roman Stanisław Ingarden, a Polish mathematical physicist, 

Vector Store Retriever

In [12]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

In [13]:
documents = [
    Document(page_content="Quantum computing leverages quantum bits, or qubits, which can represent both 0 and 1 simultaneously, allowing for parallel computation on a massive scale."),
    Document(page_content="General relativity, proposed by Albert Einstein, describes gravity not as a force but as the curvature of space-time caused by mass and energy."),
    Document(page_content="Machine learning is a subset of artificial intelligence that enables systems to learn patterns from data and make predictions or decisions without being explicitly programmed."),
    Document(page_content="The theory of evolution by natural selection, introduced by Charles Darwin, explains how species adapt and change over generations through inherited traits."),
]


In [14]:
# Step 2: Initialize embedding model
embedding_model = OpenAIEmbeddings()

# Step 3: Create Chroma vector store in memory
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name="my_collection"
)

OpenAIEmbeddings() converts your documents into numerical vectors.

Chroma.from_documents() stores these vectors in memory using Chroma for fast searching.

This setup allows you to search similar documents later based on meaning, not just keywords.

In [15]:
# Step 4: Convert vectorstore into a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [16]:
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x7b0d3e9934d0>, search_kwargs={'k': 2})

In [17]:
query = "How does quantum computing use qubits?"
results = retriever.invoke(query)

In [18]:
results

[Document(metadata={}, page_content='Quantum computing leverages quantum bits, or qubits, which can represent both 0 and 1 simultaneously, allowing for parallel computation on a massive scale.'),
 Document(metadata={}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn patterns from data and make predictions or decisions without being explicitly programmed.')]

In [19]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Quantum computing leverages quantum bits, or qubits, which can represent both 0 and 1 simultaneously, allowing for parallel computation on a massive scale.

--- Result 2 ---
Machine learning is a subset of artificial intelligence that enables systems to learn patterns from data and make predictions or decisions without being explicitly programmed.


MMR

In [20]:
docs = [
    Document(page_content="ChatGPT is a conversational AI model developed by OpenAI."),
    Document(page_content="BERT is a transformer model optimized for understanding language context."),
    Document(page_content="GPT models are used to generate human-like text."),
    Document(page_content="DALL·E is an AI model that generates images from text prompts."),
    Document(page_content="OpenAI developed both GPT and DALL·E models."),
    Document(page_content="Conversational AI is used in chatbots and virtual assistants."),
]
query = "What models are developed by OpenAI?"


In [24]:
from langchain_community.vectorstores import FAISS

# Initialize OpenAI embeddings
embedding_model = OpenAIEmbeddings()

# Step 2: Create the FAISS vector store from documents
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

In [25]:
# Enable MMR in the retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",# <-- This enables MMR
    search_kwargs={"k": 3, "lambda_mult": 0.5}  # k = top results, lambda_mult = relevance-diversity balance
)

In [26]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7b0d3d972390>, search_type='mmr', search_kwargs={'k': 3, 'lambda_mult': 0.5})

In [29]:
query = "What models are developed by OpenAI?"
results = retriever.invoke(query)

In [30]:
results

[Document(id='495ca1d7-dad9-4410-91b8-465d617db381', metadata={}, page_content='OpenAI developed both GPT and DALL·E models.'),
 Document(id='80ac244d-3442-4c28-9527-78daa0e0f858', metadata={}, page_content='Conversational AI is used in chatbots and virtual assistants.'),
 Document(id='d275de7a-ad60-43cc-842d-834180877f6d', metadata={}, page_content='BERT is a transformer model optimized for understanding language context.')]

Multiquery Retriever

In [31]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever

In [32]:
docs = [
    Document(page_content="LangChain is a powerful framework for building applications with large language models."),
    Document(page_content="LangChain makes it easy to chain prompts, memory, and tools to create LLM-powered workflows."),
    Document(page_content="Chroma is a vector database used to store and retrieve document embeddings efficiently."),
    Document(page_content="Vector embeddings are dense representations of text used for similarity search."),
    Document(page_content="MMR, or Maximal Marginal Relevance, helps retrieve results that are both relevant and diverse."),
    Document(page_content="LangChain supports various vector stores including Chroma, FAISS, and Pinecone."),
    Document(page_content="Prompt chaining in LangChain allows multi-step reasoning with LLMs."),
    Document(page_content="FAISS is a library used for fast similarity search among high-dimensional vectors."),
]

In [33]:
# Initialize OpenAI embeddings
embedding_model = OpenAIEmbeddings()

# Create FAISS vector store
vectorstore = FAISS.from_documents(documents=docs, embedding=embedding_model)

In [34]:
# Create retrievers
similarity_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [35]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 5}),
    llm=ChatOpenAI(model="gpt-3.5-turbo")
)

In [36]:
query = "How is conversational AI applied in virtual assistants and customer support?"


In [37]:
# Retrieve results
similarity_results = similarity_retriever.invoke(query)
multiquery_results= multiquery_retriever.invoke(query)

In [38]:
for i, doc in enumerate(similarity_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

print("*"*150)

for i, doc in enumerate(multiquery_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
LangChain is a powerful framework for building applications with large language models.

--- Result 2 ---
LangChain makes it easy to chain prompts, memory, and tools to create LLM-powered workflows.

--- Result 3 ---
Vector embeddings are dense representations of text used for similarity search.

--- Result 4 ---
LangChain supports various vector stores including Chroma, FAISS, and Pinecone.

--- Result 5 ---
Prompt chaining in LangChain allows multi-step reasoning with LLMs.
******************************************************************************************************************************************************

--- Result 1 ---
LangChain is a powerful framework for building applications with large language models.

--- Result 2 ---
LangChain makes it easy to chain prompts, memory, and tools to create LLM-powered workflows.

--- Result 3 ---
Vector embeddings are dense representations of text used for similarity search.

--- Result 4 ---
LangChain support

ContextualCompressionRetriever

In [40]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain_core.documents import Document

In [41]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [42]:
# Create a FAISS vector store from the documents
embedding_model = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(docs, embedding_model)

In [43]:
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [44]:
# Set up the compressor using an LLM
llm = ChatOpenAI(model="gpt-3.5-turbo")
compressor = LLMChainExtractor.from_llm(llm)

In [45]:
# Create the contextual compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [46]:
# Query the retriever
query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

In [47]:
for i, doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)



--- Result 1 ---
Photosynthesis is the process by which green plants convert sunlight into energy.

--- Result 2 ---
The chlorophyll in plant cells captures sunlight during photosynthesis.
